<a href="https://colab.research.google.com/github/Law2030/apps/blob/main/ollama_colab_free_server_en.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

An LLM server that runs Ollama on Google Colab's GPU and exposes the Ollama endpoint via an ngrok tunnel.

Key Features
- ✅ Completely free. No data sent to external APIs — privacy protected via local inference
- 🎛️ Select a model via UI; pull and launch are fully automated
- 🚀 Instantly connectable from Continue or Claude Code


> ※ Append `/v1` to the base URL for Codex and other OpenAI-compatible clients. Example: `https://xxxx.ngrok-free.app/v1`

In [11]:
#@title 📋 Model Registry

# @markdown ### Model Configuration
# @markdown > Enter the model names you want to add, separated by commas. Guide: `8B` recommended · `14B` feasible · `20B+` slow
# @markdown > - Find official model names here: https://ollama.com/search

model_list = "qwen3:8b, qwen3:14b, qwen2.5-coder:7b, qwen2.5-coder:14b, ministral-3:8b, ministral-3:14b, devstral-small-2:24b, gpt-oss:20b, deepseek-r1:8b, glm-4.7-flash, qwen3.5:9b, qwen3.5:4b, qwen3.5:2b, qwen3.5:0.8b, nemotron-3-nano:4b, nemotron-3-nano:30b" #@param {type:"string"}

# @markdown ### context length (num_ctx)
# @markdown > T4 (16GB VRAM) Guide
# @markdown > | Model Size | Rec. ctx | Notes |
# @markdown > |---|---|---|
# @markdown > | ~4B  | 32768 | Plentiful |
# @markdown > | ~8B  | 16384 | Standard |
# @markdown > | ~14B | 8192  | VRAM tight |
# @markdown > | ~20B+ | 4096 | Min. usage |
# @markdown >
# @markdown > Default (0) lets Ollama decide (equivalent to 4096).
num_ctx = 16384 #@param {type:"integer"}

AVAILABLE_MODELS = [
    model.strip()
    for model in model_list.split(',')
    if model.strip()
]

if not AVAILABLE_MODELS:
    raise ValueError("❌ Model list is empty. Please enter at least one model.")

import ipywidgets as widgets
from IPython.display import display

header = widgets.HTML(
    '<h3>📦 Select a Model</h3>'
    '<p style="margin: 5px 0 10px 0; font-size: 13px;">'
    'Select one model to launch, then run the next cell.</p>'
)

model_selector = widgets.RadioButtons(
    options=AVAILABLE_MODELS,
    value=AVAILABLE_MODELS[0],
    layout=widgets.Layout(padding='0 0 0 20px')
)

display(widgets.VBox([header, model_selector]))

print(f"✅ Model list loaded: {len(AVAILABLE_MODELS)} model(s)")
print("➡️ After selecting a model, run the next cell (Server).")


✅ Model list loaded: 16 model(s)
➡️ After selecting a model, run the next cell (Server).


In [19]:

# ============================================================
# SAUDI LAWS + OLLAMA + NGROK
# COMPLETE GOOGLE COLAB SERVER
# ============================================================

import os
import sys
import json
import time
import subprocess
import threading
import requests

from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from getpass import getpass

# ============================================================
# 1. CONFIGURATION
# ============================================================

SAUDI_LAWS_SOURCE_URL = "https://www.saudi-laws.com"

MODEL_NAME = "qwen3.5:0.8b"

OLLAMA_LOCAL_URL = "http://127.0.0.1:11434"
API_LOCAL_URL = "http://127.0.0.1:8000"

SERVER_PORT = 8000

CACHE_DIR = Path("/content/saudi_laws_cache")
LAWS_FILE = CACHE_DIR / "laws.json"
DOCS_FILE = CACHE_DIR / "documents.json"

PAGE_SIZE = 100
DOWNLOAD_WORKERS = 4
RETRY_COUNT = 3

FORCE_REFRESH = False

# ============================================================
# 2. INSTALL DEPENDENCIES
# ============================================================

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "fastapi",
        "uvicorn",
        "requests",
        "pyngrok"
    ],
    check=True
)

from fastapi import FastAPI, HTTPException, Request
from fastapi.responses import JSONResponse
import uvicorn

from pyngrok import ngrok
from pyngrok.conf import PyngrokConfig

# ============================================================
# 3. PREPARE CACHE
# ============================================================

CACHE_DIR.mkdir(parents=True, exist_ok=True)

laws = []
documents = {}

def save_laws():
    temporary = LAWS_FILE.with_suffix(".tmp")

    with open(temporary, "w", encoding="utf-8") as f:
        json.dump(laws, f, ensure_ascii=False)

    temporary.replace(LAWS_FILE)


def save_documents():
    temporary = DOCS_FILE.with_suffix(".tmp")

    with open(temporary, "w", encoding="utf-8") as f:
        json.dump(documents, f, ensure_ascii=False)

    temporary.replace(DOCS_FILE)


def load_cache():
    global laws, documents

    if LAWS_FILE.exists():
        with open(LAWS_FILE, "r", encoding="utf-8") as f:
            laws = json.load(f)

    if DOCS_FILE.exists():
        with open(DOCS_FILE, "r", encoding="utf-8") as f:
            documents = json.load(f)


if not FORCE_REFRESH:
    load_cache()

print("Cached laws:", len(laws))
print("Cached documents:", len(documents))

# ============================================================
# 4. HTTP HELPER
# ============================================================

BASE_URL = SAUDI_LAWS_SOURCE_URL.rstrip("/")

session = requests.Session()
session.headers.update({
    "Accept": "application/json",
    "User-Agent": "SaudiLegalLibrary/1.0"
})


def get_json(url):
    last_error = None

    for attempt in range(RETRY_COUNT):
        try:
            response = session.get(
                url,
                timeout=(15, 90)
            )

            response.raise_for_status()

            return response.json()

        except Exception as error:
            last_error = error
            print(
                f"Retry {attempt + 1}/{RETRY_COUNT}: {error}"
            )
            time.sleep(2 * (attempt + 1))

    raise RuntimeError(
        f"Failed to fetch {url}: {last_error}"
    )

# ============================================================
# 5. DOWNLOAD LAW LIST
# ============================================================

def download_all_laws():
    global laws

    print("\nDownloading law list from www.saudi-laws.com")

    first_url = (
        f"{BASE_URL}/api/boe-list"
        f"?page=1&pageSize={PAGE_SIZE}"
    )

    first_data = get_json(first_url)

    if not first_data.get("success"):
        raise RuntimeError(
            "Saudi-Laws API did not return success."
        )

    all_laws = list(first_data.get("collection", []))

    total_pages = int(first_data.get("totalPages", 1))

    print("Total laws:", first_data.get("totalCount"))
    print("Total pages:", total_pages)

    for page in range(2, total_pages + 1):
        url = (
            f"{BASE_URL}/api/boe-list"
            f"?page={page}&pageSize={PAGE_SIZE}"
        )

        data = get_json(url)

        all_laws.extend(data.get("collection", []))

        print(f"Law listing: {page}/{total_pages}")

    unique = {}

    for law in all_laws:
        slug = law.get("slug")

        if slug:
            unique[slug] = law

    laws = list(unique.values())

    save_laws()

    print("Law listing saved:", len(laws))

    return laws


if FORCE_REFRESH or not laws:
    download_all_laws()

# ============================================================
# 6. DOWNLOAD ALL LAW DOCUMENTS
# ============================================================

def download_document(law):
    slug = law.get("slug")

    if not slug:
        return None, None

    url = (
        f"{BASE_URL}/api/boe-doc"
        f"?slug={requests.utils.quote(slug, safe='')}"
    )

    try:
        data = get_json(url)

        if not data.get("success"):
            raise RuntimeError("Document API returned unsuccessful response")

        return slug, data

    except Exception as error:
        return slug, {
            "_download_error": str(error),
            "slug": slug,
            "title": law.get("title", ""),
            "articles": [],
            "blocks": []
        }


def download_all_documents():
    global documents

    missing = [
        law for law in laws
        if (
            law.get("slug") not in documents
            or documents[law.get("slug")].get("_download_error")
        )
    ]

    print("\nDocuments cached:", len(documents))
    print("Documents remaining:", len(missing))

    if not missing:
        print("All law documents already cached.")
        return

    completed = 0

    with ThreadPoolExecutor(
        max_workers=DOWNLOAD_WORKERS
    ) as executor:

        futures = [
            executor.submit(download_document, law)
            for law in missing
        ]

        for future in as_completed(futures):
            slug, data = future.result()

            if slug and data:
                documents[slug] = data

            completed += 1

            if completed % 10 == 0 or completed == len(missing):
                save_documents()

                print(
                    f"Documents downloaded: "
                    f"{completed}/{len(missing)}"
                )

    save_documents()

    failed = sum(
        1
        for doc in documents.values()
        if doc.get("_download_error")
    )

    print("\nDocument download completed.")
    print("Cached documents:", len(documents))
    print("Failed documents:", failed)


download_all_documents()

# ============================================================
# 7. LIBRARY DIAGNOSTICS
# ============================================================

total_articles = sum(
    len(doc.get("articles", []))
    for doc in documents.values()
)

print("\n======================================")
print("SAUDI LEGAL LIBRARY")
print("======================================")
print("Laws:", len(laws))
print("Documents:", len(documents))
print("Articles:", total_articles)
print("======================================")

# ============================================================
# 8. INSTALL OLLAMA
# ============================================================

if not Path("/usr/local/bin/ollama").exists():
    print("Installing Ollama...")

    subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh",
        shell=True,
        check=True
    )

# ============================================================
# 9. START OLLAMA
# ============================================================

ollama_env = os.environ.copy()

ollama_env.update({
    "OLLAMA_HOST": "127.0.0.1:11434",
    "OLLAMA_KEEP_ALIVE": "30m",
    "OLLAMA_NUM_PARALLEL": "1",
    "OLLAMA_MAX_LOADED_MODELS": "1",
    "OLLAMA_FLASH_ATTENTION": "1",
    "OLLAMA_KV_CACHE_TYPE": "q8_0"
})

print("\nStarting Ollama...")

ollama_log = open("/content/ollama.log", "w")

ollama_process = subprocess.Popen(
    ["/usr/local/bin/ollama", "serve"],
    env=ollama_env,
    stdout=ollama_log,
    stderr=subprocess.STDOUT
)

ollama_ready = False

for attempt in range(60):
    try:
        response = requests.get(
            f"{OLLAMA_LOCAL_URL}/api/tags",
            timeout=3
        )

        if response.ok:
            ollama_ready = True
            break

    except requests.RequestException:
        pass

    time.sleep(2)

if not ollama_ready:
    raise RuntimeError(
        "Ollama failed to start. Inspect /content/ollama.log"
    )

print("Ollama server is ready.")

# ============================================================
# 10. DOWNLOAD MODEL
# ============================================================

print("Loading model:", MODEL_NAME)

subprocess.run(
    [
        "/usr/local/bin/ollama",
        "pull",
        MODEL_NAME
    ],
    check=True
)

# ============================================================
# 11. WARM UP MODEL
# ============================================================

print("Warming up model...")

warmup = requests.post(
    f"{OLLAMA_LOCAL_URL}/api/generate",
    json={
        "model": MODEL_NAME,
        "prompt": 'Return only {"status":"ok"}',
        "stream": False,
        "keep_alive": "30m",
        "options": {
            "temperature": 0,
            "num_ctx": 4096,
            "num_predict": 20
        }
    },
    timeout=180
)

warmup.raise_for_status()

print("Model warmed up.")

# ============================================================
# 12. CREATE FASTAPI SERVER
# ============================================================

app = FastAPI(
    title="Saudi Legal Library and Ollama"
)

# ============================================================
# HEALTH
# ============================================================

@app.get("/")
def home():
    return {
        "success": True,
        "message": "Saudi legal and Ollama server running",
        "laws": len(laws),
        "documents": len(documents),
        "model": MODEL_NAME
    }


@app.get("/health")
def health():
    return {
        "success": True,
        "laws": len(laws),
        "documents": len(documents),
        "articles": sum(
            len(doc.get("articles", []))
            for doc in documents.values()
        ),
        "model": MODEL_NAME
    }

# ============================================================
# BOE LAW LIST
# ============================================================

@app.get("/api/boe-list")
def boe_list(
    page: int = 1,
    pageSize: int = 12,
    q: str = "",
    type: str = "",
    status: str = "",
    facets: int = 0
):
    filtered = list(laws)

    if q:
        query = q.strip().lower()

        filtered = [
            law for law in filtered
            if (
                query in str(law.get("title", "")).lower()
                or query in str(law.get("statuteName", "")).lower()
                or query in str(law.get("summary", "")).lower()
            )
        ]

    if type:
        filtered = [
            law for law in filtered
            if str(law.get("documentType", "")).lower()
            == type.lower()
        ]

    if status:
        filtered = [
            law for law in filtered
            if str(law.get("status", "")).lower()
            == status.lower()
        ]

    total = len(filtered)

    pageSize = max(1, min(pageSize, 100))
    page = max(1, page)

    start = (page - 1) * pageSize

    return {
        "success": True,
        "collection": filtered[start:start + pageSize],
        "totalCount": total,
        "totalPages": max(1, (total + pageSize - 1) // pageSize),
        "page": page
    }

# ============================================================
# BOE LAW DOCUMENT
# ============================================================

@app.get("/api/boe-doc")
def boe_doc(slug: str):
    if not slug:
        raise HTTPException(
            status_code=400,
            detail="slug is required"
        )

    doc = documents.get(slug)

    if not doc:
        raise HTTPException(
            status_code=404,
            detail="Law document not found in cache"
        )

    if doc.get("_download_error"):
        raise HTTPException(
            status_code=503,
            detail=doc["_download_error"]
        )

    return doc

# ============================================================
# OLLAMA CHAT PROXY
# ============================================================

@app.post("/api/chat")
async def ollama_chat(request: Request):
    try:
        payload = await request.json()

        payload["model"] = payload.get("model") or MODEL_NAME

        payload.setdefault("keep_alive", "30m")

        response = requests.post(
            f"{OLLAMA_LOCAL_URL}/api/chat",
            json=payload,
            timeout=(15, 240)
        )

        try:
            response_data = response.json()
        except ValueError:
            return JSONResponse(
                status_code=502,
                content={
                    "error": "Ollama returned invalid JSON.",
                    "details": response.text[:1000]
                }
            )

        return JSONResponse(
            content=response_data,
            status_code=response.status_code
        )

    except requests.Timeout:
        return JSONResponse(
            status_code=504,
            content={
                "error": "Ollama request timed out."
            }
        )

    except requests.RequestException as error:
        return JSONResponse(
            status_code=502,
            content={
                "error": f"Ollama connection failed: {str(error)}"
            }
        )

    except Exception as error:
        return JSONResponse(
            status_code=500,
            content={
                "error": str(error)
            }
        )

# ============================================================
# 13. START FASTAPI
# ============================================================

def run_api():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=SERVER_PORT,
        log_level="warning"
    )


api_thread = threading.Thread(
    target=run_api,
    daemon=True
)

api_thread.start()

api_ready = False

for attempt in range(30):
    try:
        response = requests.get(
            API_LOCAL_URL + "/health",
            timeout=3
        )

        if response.ok:
            api_ready = True
            break

    except requests.RequestException:
        pass

    time.sleep(1)

if not api_ready:
    raise RuntimeError("FastAPI failed to start.")

print("FastAPI server is ready.")

# ============================================================
# 14. NGROK AUTHENTICATION
# ============================================================

print("\nEnter your ngrok authentication token:")

NGROK_TOKEN = getpass("Ngrok token: ").strip()

if not NGROK_TOKEN:
    raise ValueError("Ngrok authentication token is required.")

# IMPORTANT:
# Do NOT call ngrok.set_auth_token().
# Pass the token directly through PyngrokConfig.
# This avoids the existing ngrok.yml token conflict.

ngrok_config = PyngrokConfig(
    auth_token=NGROK_TOKEN,
    monitor_thread=True
)

# Close any existing default tunnel process.
try:
    ngrok.kill()
except Exception:
    pass

# ============================================================
# 15. CREATE NGROK TUNNEL
# ============================================================

print("\nCreating ngrok tunnel...")

tunnel = ngrok.connect(
    addr=SERVER_PORT,
    proto="http",
    pyngrok_config=ngrok_config
)

PUBLIC_URL = tunnel.public_url.rstrip("/")

# ============================================================
# 16. TEST PUBLIC ENDPOINTS
# ============================================================

print("\nTesting public endpoints...")

health_test = requests.get(
    PUBLIC_URL + "/health",
    timeout=30
)

health_test.raise_for_status()

list_test = requests.get(
    PUBLIC_URL + "/api/boe-list?page=1&pageSize=5",
    timeout=30
)

list_test.raise_for_status()

if not list_test.json().get("success"):
    raise RuntimeError("Public law-list test failed.")

chat_test = requests.post(
    PUBLIC_URL + "/api/chat",
    json={
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "user",
                "content": "Reply with OK only."
            }
        ],
        "stream": False,
        "options": {
            "temperature": 0,
            "num_predict": 10
        }
    },
    timeout=180
)

chat_test.raise_for_status()

# ============================================================
# 17. FINAL OUTPUT
# ============================================================

print("\n============================================")
print("SAUDI LEGAL + OLLAMA SERVER IS READY")
print("============================================")

print("PUBLIC URL:", PUBLIC_URL)

print("\nLEGAL API:")
print(PUBLIC_URL + "/api/boe-list")
print(PUBLIC_URL + "/api/boe-doc?slug=YOUR_REAL_SLUG")

print("\nOLLAMA:")
print(PUBLIC_URL + "/api/chat")

print("\nHEALTH:")
print(PUBLIC_URL + "/health")

print("\nLIBRARY:")
print("Laws:", len(laws))
print("Documents:", len(documents))
print("Articles:", total_articles)

print("\nMODEL:", MODEL_NAME)

print("\nNEXT.JS ENVIRONMENT VARIABLES:")
print("SAUDI_LAWS_URL=" + PUBLIC_URL)
print("OLLAMA_URL=" + PUBLIC_URL)
print("OLLAMA_MODEL=" + MODEL_NAME)

print("\n============================================")
print("KEEP THIS COLAB CELL RUNNING.")
print("============================================")


Cached laws: 507
Cached documents: 507

Documents cached: 507
Documents remaining: 0
All law documents already cached.

SAUDI LEGAL LIBRARY
Laws: 507
Documents: 507
Articles: 16472

Starting Ollama...
Ollama server is ready.
Loading model: qwen3.5:0.8b
Warming up model...
Model warmed up.


ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use


FastAPI server is ready.

Enter your ngrok authentication token:
Ngrok token: ··········

Creating ngrok tunnel...

Testing public endpoints...

SAUDI LEGAL + OLLAMA SERVER IS READY
PUBLIC URL: https://obtrusive-nebulizer-maturing.ngrok-free.dev

LEGAL API:
https://obtrusive-nebulizer-maturing.ngrok-free.dev/api/boe-list
https://obtrusive-nebulizer-maturing.ngrok-free.dev/api/boe-doc?slug=YOUR_REAL_SLUG

OLLAMA:
https://obtrusive-nebulizer-maturing.ngrok-free.dev/api/chat

HEALTH:
https://obtrusive-nebulizer-maturing.ngrok-free.dev/health

LIBRARY:
Laws: 507
Documents: 507
Articles: 16472

MODEL: qwen3.5:0.8b

NEXT.JS ENVIRONMENT VARIABLES:
SAUDI_LAWS_URL=https://obtrusive-nebulizer-maturing.ngrok-free.dev
OLLAMA_URL=https://obtrusive-nebulizer-maturing.ngrok-free.dev
OLLAMA_MODEL=qwen3.5:0.8b

KEEP THIS COLAB CELL RUNNING.
